# 03 · Baseline experiment — Milestone 1  *(branch `exp/03-baseline`)*

The reference point for everything later (brief §3.2). All numbers are **test period, net of costs**
(Binance spot taker fee 0.10 % + 0.02 % slippage per side, fills at the next bar's midpoint).

| Baseline | Here |
|---|---|
| Buy & Hold | enter long at the first test bar (entry cost paid), hold to the end |
| Random policy | uniform random action every decision, 5 seeds |
| PPO, price-only | configuration **A** (`[position, time-in-position] + 8 RNN factors`, dim 10), the notebook's PPO hyperparameters, 5 seeds |

Every PPO run: train on train → pick the checkpoint with the best **validation** Sharpe → one test backtest.
Runtime: ~10–25 min per PPO seed on a T4 (2 M transitions). Finished runs are skipped on re-run.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/03-baseline"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "03_baseline")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run_seeds
from rltrade import results as R
SEEDS = [0, 1, 2, 3, 4]
TOTAL_STEPS = 2_000_000             # notebook: TOTAL_STEPS = 2e6
EVAL_EVERY = 100_000
F = paths.factor_dir(L, prefer_selected=False)      # Milestone 1 uses the default RNN from 02
ENV_A = EnvConfig(state="A")
print(ENV_A)

In [ ]:
run_seeds(RunSpec("03_baseline", "buy_and_hold", env=ENV_A), [0], L.prepared, F, L.runs)      # deterministic: one run
run_seeds(RunSpec("03_baseline", "random", env=ENV_A), SEEDS, L.prepared, F, L.runs)
run_seeds(RunSpec("03_baseline", "ppo", env=ENV_A, total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY),
          SEEDS, L.prepared, F, L.runs)

## Results

In [ ]:
import matplotlib.pyplot as plt
df = R.load_results(L.runs); df = df[df.experiment == "03_baseline"]
sm = R.summarise(df)
table = R.markdown_table(sm, group=("agent", "state"))
print(table)
(L.report / "milestone1_baseline.md").write_text("# Milestone 1 - baselines (test, net of costs)\n\n" + table + "\n", encoding="utf-8")
groups = {"Buy & hold": df[df.agent == "buy_and_hold"], "Random": df[df.agent == "random"], "PPO price-only (A)": df[df.agent == "ppo"]}
R.plot_curves(df, L.runs, groups, path=L.report / "milestone1_equity.png"); plt.show()
R.plot_curves(df, L.runs, groups, kind="drawdown", path=L.report / "milestone1_drawdown.png"); plt.show()

## Milestone 1 summary (fill in, half a page)
- What was run (branch, commit, seeds, budget, costs):
- Results (table above):
- Anything unexpected (e.g. PPO never trades, trades constantly, beats / loses to buy & hold):
- Leakage check status (01 tests + LEAKAGE_AUDIT.md):